# Cuando el camino más corto no es el mejor — Modelado del terreno y función de costo

**Problema:** evacuar una zona inundada hacia un refugio. La ruta más corta puede cruzar zonas de alto riesgo.

**Pregunta eje:** *Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?*
Comparando el costo acumulado $J(P)=\sum C(u,v)$, no la longitud.

## Función de costo
$$C(u,v)=\begin{cases} d(u,v)\,\bigl(1+\alpha\,R(v)^{\beta}\bigr) & v \text{ transitable}\\ \infty & R(v)=-1\end{cases}$$

- $d(u,v)$: 1 (ortogonal) o $\sqrt2$ (diagonal).
- $R(v)$: nivel de riesgo de la celda destino (0 seco, 1–3 anegación, -1 bloqueo total).
- $\alpha$: aversión al riesgo. $\beta$: exponente no lineal.

| Nivel | Significado | Factor (α=2, β=1.5) |
|---|---|---|
| 0 | Seco | 1.00 |
| 1 | Leve | 3.00 |
| 2 | Moderado | 6.66 |
| 3 | Severo | 11.39 |
| -1 | Bloqueo | ∞ |


In [1]:
from __future__ import annotations

import math

import numpy as np

Node = tuple[int, int]


class GridEnvironment:
    """Cuadrícula de riesgo de inundación con función de costo de transición.

    Niveles: 0 seco, 1-3 anegación creciente, -1 bloqueo total.
    Costo: C(u, v) = d(u, v) * (1 + alpha * R(v) ** beta); inf si v está bloqueada.
    """

    BLOCKED = -1
    MAX_RISK = 3
    _ORTHO = ((-1, 0), (1, 0), (0, -1), (0, 1))
    _DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, rows: int, cols: int, alpha: float = 2.0,
                 beta: float = 1.5, allow_diagonal: bool = False) -> None:
        if rows <= 0 or cols <= 0:
            raise ValueError("rows y cols deben ser positivos")
        if alpha < 0 or beta < 1:
            raise ValueError("se requiere alpha >= 0 y beta >= 1")
        self.rows = rows
        self.cols = cols
        self.alpha = alpha
        self.beta = beta
        self.allow_diagonal = allow_diagonal
        self.grid: np.ndarray = np.zeros((rows, cols), dtype=np.int8)

    # ---- validación ----
    def in_bounds(self, r: int, c: int) -> bool:
        """True si (r, c) está dentro de la cuadrícula."""
        return 0 <= r < self.rows and 0 <= c < self.cols

    def _check_level(self, risk_level: int) -> None:
        if risk_level != self.BLOCKED and not 0 <= risk_level <= self.MAX_RISK:
            raise ValueError(f"risk_level inválido: {risk_level} (use -1 o 0..3)")

    # ---- modificación del terreno ----
    def set_cell(self, r: int, c: int, risk_level: int) -> None:
        """Asigna el nivel de riesgo de una celda."""
        if not self.in_bounds(r, c):
            raise IndexError(f"celda fuera de límites: ({r}, {c})")
        self._check_level(risk_level)
        self.grid[r, c] = risk_level

    def set_region_risk(self, r_start: int, r_end: int, c_start: int,
                        c_end: int, risk_level: int) -> None:
        """Asigna un nivel a la región rectangular [r_start, r_end] x [c_start, c_end] (inclusive)."""
        if r_start > r_end or c_start > c_end:
            raise ValueError("rango de región invertido")
        if not (self.in_bounds(r_start, c_start) and self.in_bounds(r_end, c_end)):
            raise IndexError("región fuera de límites")
        self._check_level(risk_level)
        self.grid[r_start:r_end + 1, c_start:c_end + 1] = risk_level

    def load_matrix(self, matrix: np.ndarray | list[list[int]]) -> None:
        """Carga una matriz 2D completa validando dimensiones y niveles (-1 o 0..3)."""
        arr = np.asarray(matrix, dtype=np.int8)
        if arr.shape != (self.rows, self.cols):
            raise ValueError(f"dimensiones inválidas: se esperaba {(self.rows, self.cols)}, se recibió {arr.shape}")
        if not np.isin(arr, (-1, 0, 1, 2, 3)).all():
            raise ValueError("la matriz contiene niveles fuera de {-1, 0, 1, 2, 3}")
        self.grid[:, :] = arr

    def reset(self) -> None:
        """Limpia el terreno a ceros (todo seco)."""
        self.grid.fill(0)

    # ---- consultas ----
    def is_blocked(self, r: int, c: int) -> bool:
        """True si la celda es intransitable (bloqueada o fuera de límites)."""
        return (not self.in_bounds(r, c)) or self.grid[r, c] == self.BLOCKED

    def cost(self, u: Node, v: Node) -> float:
        """Costo de transición u -> v (v adyacente). inf si u o v están bloqueadas."""
        # [FIX - Andrey Lechuga (Módulo 3)]:
        # Se agregaron validaciones de límites para `u` y `v` (IndexError), chequeo de
        # bloqueo en el origen `u`, y restricción de diagonal cuando `allow_diagonal=False`.
        if not (self.in_bounds(*u) and self.in_bounds(*v)):
            raise IndexError(f"coordenadas fuera de límites: u={u}, v={v}")
        dr, dc = abs(u[0] - v[0]), abs(u[1] - v[1])
        if max(dr, dc) != 1:
            raise ValueError(f"{u} y {v} no son adyacentes")
        if dr + dc == 2 and not self.allow_diagonal:
            raise ValueError(f"movimiento diagonal no permitido (allow_diagonal=False): {u} -> {v}")
        if self.is_blocked(*u) or self.is_blocked(*v):
            return math.inf
        d = math.sqrt(2.0) if dr + dc == 2 else 1.0
        risk = float(self.grid[v])
        return d * (1.0 + self.alpha * risk ** self.beta)

    def get_neighbors(self, node: Node) -> list[tuple[Node, float]]:
        """Vecinos transitables de `node` como ((r, c), costo). Nunca incluye bloqueados."""
        r, c = node
        if not self.in_bounds(r, c):
            raise IndexError(f"nodo fuera de límites: {node}")
        result: list[tuple[Node, float]] = []
        for dr, dc in self._ORTHO:
            nr, nc = r + dr, c + dc
            if not self.is_blocked(nr, nc):
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        if self.allow_diagonal:
            for dr, dc in self._DIAG:
                nr, nc = r + dr, c + dc
                if self.is_blocked(nr, nc):
                    continue
                # Anti corner-cutting: impide diagonal cuando las dos ortogonales intermedias están bloqueadas
                if self.is_blocked(r + dr, c) and self.is_blocked(r, c + dc):
                    continue
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        return result

In [2]:
# Smoke test: mapa 5x5 con un bloqueo
env = GridEnvironment(5, 5)
env.set_cell(2, 2, -1)
assert env.is_blocked(2, 2)
nb = dict(env.get_neighbors((2, 1)))
assert (2, 2) not in nb and set(nb) == {(1, 1), (3, 1), (2, 0)}, nb
print("Smoke test OK:", nb)

# Costos ponderados (alpha=2, beta=1.5)
env.reset()
env.set_cell(1, 0, 1)
env.set_cell(0, 1, 3)
assert math.isclose(env.cost((0, 0), (1, 0)), 1 + 2 * 1 ** 1.5)      # 3.0
assert math.isclose(env.cost((0, 0), (0, 1)), 1 + 2 * 3 ** 1.5)      # ~11.39
assert env.cost((0, 0), (1, 0)) < env.cost((0, 0), (0, 1))           # más riesgo, más costo
env.set_cell(0, 1, -1)
assert env.cost((0, 0), (0, 1)) == math.inf
assert (0, 1) not in dict(env.get_neighbors((0, 0)))

# Región, carga matricial y reset
env.set_region_risk(0, 1, 0, 1, 2)
assert (env.grid[0:2, 0:2] == 2).all()
env.load_matrix(np.zeros((5, 5), dtype=np.int8))
assert (env.grid == 0).all()
env.reset()
assert (env.grid == 0).all()

# Diagonales: costo sqrt(2) y anti corner-cutting
d = GridEnvironment(3, 3, allow_diagonal=True)
assert math.isclose(dict(d.get_neighbors((1, 1)))[(0, 0)], math.sqrt(2))
d.set_cell(0, 1, -1)
d.set_cell(1, 0, -1)
assert (0, 0) not in dict(d.get_neighbors((1, 1)))   # ambas bloqueadas -> impide corte de esquina
d.set_cell(1, 0, 0)
assert (0, 0) in dict(d.get_neighbors((1, 1)))       # solo una bloqueada -> permitido

# Validaciones
for bad in (lambda: env.set_cell(9, 9, 0), lambda: env.set_cell(0, 0, 7)):
    try:
        bad()
        raise AssertionError("debió fallar")
    except (IndexError, ValueError):
        pass
print("Todas las pruebas pasaron")


Smoke test OK: {(1, 1): 1.0, (3, 1): 1.0, (2, 0): 1.0}
Todas las pruebas pasaron


## Módulo 2: Algoritmos de búsqueda (Dijkstra y A*) y conteo de nodos explorados

### 1. Comparación técnica: Dijkstra vs A*
Ambos algoritmos resuelven el problema de encontrar una ruta óptima desde el origen `start` hasta el refugio `goal` minimizando el **costo acumulado** $J(P) = \sum_{(u,v) \in P} C(u,v)$ sobre el grafo implícito generado por `GridEnvironment.get_neighbors(u)`:

- **Algoritmo de Dijkstra (búsqueda de costo uniforme):** Ordena la frontera de exploración (cola de prioridad) únicamente por el costo acumulado desde el inicio, $f(n) = g(n)$. Dado que todas las celdas transitables tienen costo estrictamente positivo $C(u,v) \ge d(u,v) \ge 1 > 0$, la primera vez que un nodo se extrae de la cola de prioridad su costo $g(n)$ es óptimo. Sin embargo, al carecer de información direccional hacia la meta, expande contornos de igual costo en todas las direcciones.
- **Algoritmo A* (búsqueda informada):** Incorpora una heurística $h(n)$ que estima el costo mínimo restante desde $n$ hasta `goal`, priorizando los nodos mediante $f(n) = g(n) + h(n)$. Esto focaliza la búsqueda hacia el refugio y reduce los nodos expandidos (`nodes_explored`) garantizando el mismo costo óptimo que Dijkstra siempre que $h(n)$ sea admisible.

### 2. Definición formal de la heurística admisible $h(n)$
Sea $n = (r_n, c_n)$ el nodo actual y $g = (r_g, c_g)$ el refugio (`goal`), con $\Delta r = |r_n - r_g|$ y $\Delta c = |c_n - c_g|$. La heurística se adapta a la topología del entorno:

1. **Sin diagonales (`allow_diagonal = False`) — Distancia Manhattan:**
   $$h_{\text{Manhattan}}(n, g) = \Delta r + \Delta c$$
2. **Con diagonales (`allow_diagonal = True`) — Distancia Octile / Euclidiana:**
   $$h_{\text{Octile}}(n, g) = \max(\Delta r, \Delta c) + (\sqrt{2} - 1)\,\min(\Delta r, \Delta c)$$
   *(Nota: la distancia Euclidiana $h_{\text{Euclidiana}}(n, g) = \sqrt{\Delta r^2 + \Delta c^2}$ también es admisible al cumplir $h_{\text{Euclidiana}}(n, g) \le h_{\text{Octile}}(n, g)$; la distancia Octile representa la distancia mínima exacta en una cuadrícula 8-conexa con pasos ortogonales de costo $1$ y diagonales de costo $\sqrt{2}$).*

### 3. Justificación matemática de admisibilidad ($h(n) \le h^*(n)$)
Para toda celda transitable $v$, el riesgo cumple $R(v) \in \{0, 1, 2, 3\}$, con $\alpha \ge 0$ y $\beta \ge 1$. Por consiguiente, el factor de penalización por inundación verifica siempre:
$$1 + \alpha\,R(v)^{\beta} \ge 1$$
Al multiplicar por la distancia base $d(u,v) \in \{1, \sqrt{2}\}$, cada transición transitable satisface:
$$C(u, v) = d(u, v)\,\bigl(1 + \alpha\,R(v)^{\beta}\bigr) \ge d(u, v)$$
Sumando sobre cualquier camino válido $P$ de $n$ a $g$, el costo real acumulado cumple:
$$J(P) = \sum_{(u,v) \in P} C(u,v) \ge \sum_{(u,v) \in P} d(u,v) \ge h(n, g)$$
Tomando el mínimo sobre todos los caminos posibles se obtiene $h(n, g) \le h^*(n, g)$ (**admisibilidad**) y $h(u, g) \le d(u,v) + h(v, g) \le C(u,v) + h(v, g)$ (**consistencia / monotonía**). Por lo tanto, $h(n)$ **nunca sobreestima** el costo real al refugio y A* encuentra garantizadamente la ruta óptima.


In [3]:
from dataclasses import dataclass
import heapq
import time


@dataclass
class SearchResult:
    """Resultado estándar de búsqueda para consumo de pruebas y visualización.

    Permite acceso tanto por atributos (`res.cost`) como por llave de diccionario
    (`res["cost"]`) para facilitar la integración con los demás módulos.

    [FIX - Andrey Lechuga (Módulo 3)]:
    Se conserva intacto `to_dict()` con las 4 llaves de Martín (`path`, `cost`,
    `nodes_explored`, `visited_order`) y se añaden propiedades/llaves de compatibilidad
    (`time_ms`, `found`, `nodes_expanded`, `steps`) para integrar sin fricción con
    los módulos de Escenarios (Módulo 3), Visualización (Módulo 4) y Documentación (Módulo 5).
    """

    path: list[Node]
    cost: float
    nodes_explored: int
    visited_order: list[Node]
    time_ms: float = 0.0

    @property
    def found(self) -> bool:
        return len(self.path) > 0 and math.isfinite(self.cost)

    @property
    def nodes_expanded(self) -> int:
        return self.nodes_explored

    @property
    def steps(self) -> int:
        return max(0, len(self.path) - 1)

    def __getitem__(self, key: str) -> list[Node] | float | int | bool:
        if key in ("path", "cost", "nodes_explored", "visited_order", "time_ms", "found", "nodes_expanded", "steps"):
            return getattr(self, key)
        raise KeyError(key)

    def __contains__(self, key: object) -> bool:
        return key in ("path", "cost", "nodes_explored", "visited_order", "time_ms", "found", "nodes_expanded", "steps")

    def keys(self) -> tuple[str, str, str, str]:
        return ("path", "cost", "nodes_explored", "visited_order")

    def get(self, key: str, default: object = None) -> object:
        return getattr(self, key, default) if key in self else default

    def to_dict(self) -> dict[str, list[Node] | float | int]:
        return {
            "path": self.path,
            "cost": self.cost,
            "nodes_explored": self.nodes_explored,
            "visited_order": self.visited_order,
        }


def heuristic(node: Node, goal: Node, allow_diagonal: bool = False) -> float:
    """Heurística admisible y consistente h(n) según la conectividad del entorno.

    - allow_diagonal=False: distancia Manhattan.
    - allow_diagonal=True: distancia Octile (cota inferior en malla 8-conexa).
    """
    dr = abs(node[0] - goal[0])
    dc = abs(node[1] - goal[1])
    if not allow_diagonal:
        return float(dr + dc)
    return max(dr, dc) + (math.sqrt(2.0) - 1.0) * min(dr, dc)


def _reconstruct_path(came_from: dict[Node, Node], start: Node, goal: Node) -> list[Node]:
    """Reconstruye la ruta ordenada desde `start` hasta `goal`."""
    path: list[Node] = [goal]
    current = goal
    while current != start:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


def dijkstra(env: GridEnvironment, start: Node, goal: Node) -> SearchResult:
    """Algoritmo de Dijkstra (costo uniforme) usando la interfaz de GridEnvironment."""
    t0 = time.perf_counter()
    if env.is_blocked(*start) or env.is_blocked(*goal):
        return SearchResult(
            path=[], cost=math.inf, nodes_explored=0, visited_order=[],
            time_ms=(time.perf_counter() - t0) * 1000.0,
        )

    open_heap: list[tuple[float, int, Node]] = [(0.0, 0, start)]
    came_from: dict[Node, Node] = {}
    g_score: dict[Node, float] = {start: 0.0}
    closed: set[Node] = set()
    visited_order: list[Node] = []
    nodes_explored = 0
    tie_breaker = 0

    while open_heap:
        current_cost, _, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)
        visited_order.append(current)
        nodes_explored += 1

        if current == goal:
            return SearchResult(
                path=_reconstruct_path(came_from, start, goal),
                cost=current_cost,
                nodes_explored=nodes_explored,
                visited_order=visited_order,
                time_ms=(time.perf_counter() - t0) * 1000.0,
            )

        for neighbor, step_cost in env.get_neighbors(current):
            if neighbor in closed:
                continue
            tentative_g = current_cost + step_cost
            if tentative_g < g_score.get(neighbor, math.inf):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                tie_breaker += 1
                heapq.heappush(open_heap, (tentative_g, tie_breaker, neighbor))

    return SearchResult(
        path=[], cost=math.inf, nodes_explored=nodes_explored, visited_order=visited_order,
        time_ms=(time.perf_counter() - t0) * 1000.0,
    )


def a_star(env: GridEnvironment, start: Node, goal: Node, heuristic_fn=None) -> SearchResult:
    """Algoritmo A* con heurística admisible adaptada a `env.allow_diagonal`."""
    t0 = time.perf_counter()
    if env.is_blocked(*start) or env.is_blocked(*goal):
        return SearchResult(
            path=[], cost=math.inf, nodes_explored=0, visited_order=[],
            time_ms=(time.perf_counter() - t0) * 1000.0,
        )

    h_fn = heuristic_fn or (lambda u, g: heuristic(u, g, env.allow_diagonal))
    h_start = h_fn(start, goal)
    open_heap: list[tuple[float, float, int, Node]] = [(h_start, h_start, 0, start)]
    came_from: dict[Node, Node] = {}
    g_score: dict[Node, float] = {start: 0.0}
    closed: set[Node] = set()
    visited_order: list[Node] = []
    nodes_explored = 0
    tie_breaker = 0

    while open_heap:
        _, _, _, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)
        visited_order.append(current)
        nodes_explored += 1

        if current == goal:
            return SearchResult(
                path=_reconstruct_path(came_from, start, goal),
                cost=g_score[current],
                nodes_explored=nodes_explored,
                visited_order=visited_order,
                time_ms=(time.perf_counter() - t0) * 1000.0,
            )

        for neighbor, step_cost in env.get_neighbors(current):
            if neighbor in closed:
                continue
            tentative_g = g_score[current] + step_cost
            if tentative_g < g_score.get(neighbor, math.inf):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                h_val = h_fn(neighbor, goal)
                f_val = tentative_g + h_val
                tie_breaker += 1
                heapq.heappush(open_heap, (f_val, h_val, tie_breaker, neighbor))

    return SearchResult(
        path=[], cost=math.inf, nodes_explored=nodes_explored, visited_order=visited_order,
        time_ms=(time.perf_counter() - t0) * 1000.0,
    )

In [4]:
# Smoke Test de Búsqueda (Módulo 2: Dijkstra y A*)

# 1. Ruta recta con riesgo alto (nivel 3) vs rodeo por celdas secas (nivel 0)
env_rodeo = GridEnvironment(5, 5, alpha=2.0, beta=1.5, allow_diagonal=False)
# Línea directa en fila 0 ((0,1), (0,2), (0,3)) con anegación severa (nivel 3)
env_rodeo.set_region_risk(0, 0, 1, 3, 3)
start_node, goal_node = (0, 0), (0, 4)

res_dijkstra_rodeo = dijkstra(env_rodeo, start_node, goal_node)
res_astar_rodeo = a_star(env_rodeo, start_node, goal_node)

# Costo directo por fila 0 sería: 3 * (1 + 2 * 3^1.5) + 1 ≈ 35.18 (4 pasos)
# Rodeo seco por fila 1 toma 6 pasos de costo 1.0 -> costo total = 6.0
direct_risky_cost = sum(env_rodeo.cost((0, c), (0, c + 1)) for c in range(4))
assert math.isclose(res_dijkstra_rodeo["cost"], 6.0), res_dijkstra_rodeo
assert math.isclose(res_astar_rodeo["cost"], 6.0), res_astar_rodeo
assert res_dijkstra_rodeo["cost"] < direct_risky_cost
assert res_dijkstra_rodeo["path"] == res_astar_rodeo["path"]
assert all(env_rodeo.grid[r, c] == 0 for r, c in res_astar_rodeo["path"])
print(
    f"Prueba 1 OK (Rodeo seguro): costo óptimo = {res_astar_rodeo['cost']:.2f} "
    f"vs ruta recta riesgosa = {direct_risky_cost:.2f} | pasos = {len(res_astar_rodeo['path']) - 1}"
)

# 2. Comparación de nodes_explored entre Dijkstra y A* (ortogonal y diagonal)
env_comp = GridEnvironment(10, 10, alpha=2.0, beta=1.5, allow_diagonal=False)
env_comp.set_region_risk(2, 6, 3, 5, 2)  # zona de riesgo moderado
env_comp.set_region_risk(4, 4, 1, 6, 3)  # franja de riesgo severo

res_d = dijkstra(env_comp, (0, 0), (8, 8))
res_a = a_star(env_comp, (0, 0), (8, 8))
assert math.isclose(res_d["cost"], res_a["cost"])
assert res_a["nodes_explored"] <= res_d["nodes_explored"]
assert len(res_a["visited_order"]) == res_a["nodes_explored"]
print(
    f"Prueba 2a OK (Ortogonal): costo Dijkstra = {res_d['cost']:.2f}, A* = {res_a['cost']:.2f} | "
    f"explorados Dijkstra = {res_d['nodes_explored']}, A* = {res_a['nodes_explored']}"
)

env_diag = GridEnvironment(10, 10, alpha=2.0, beta=1.5, allow_diagonal=True)
env_diag.set_region_risk(2, 6, 3, 5, 2)
res_d_diag = dijkstra(env_diag, (0, 0), (8, 8))
res_a_diag = a_star(env_diag, (0, 0), (8, 8))
assert math.isclose(res_d_diag["cost"], res_a_diag["cost"])
assert res_a_diag["nodes_explored"] <= res_d_diag["nodes_explored"]
print(
    f"Prueba 2b OK (Diagonal):  costo Dijkstra = {res_d_diag['cost']:.2f}, A* = {res_a_diag['cost']:.2f} | "
    f"explorados Dijkstra = {res_d_diag['nodes_explored']}, A* = {res_a_diag['nodes_explored']}"
)

# 3. Destino encerrado por bloqueos (-1) y nodos inicio/meta bloqueados
env_blocked = GridEnvironment(5, 5, allow_diagonal=True)
# Encerramos el refugio (4, 4) con celdas bloqueadas (-1)
for r_b, c_b in ((3, 4), (4, 3), (3, 3)):
    env_blocked.set_cell(r_b, c_b, -1)

res_d_enc = dijkstra(env_blocked, (0, 0), (4, 4))
res_a_enc = a_star(env_blocked, (0, 0), (4, 4))
assert res_d_enc["path"] == [] and res_d_enc["cost"] == math.inf
assert res_a_enc["path"] == [] and res_a_enc["cost"] == math.inf

# Inicio o destino directamente bloqueado
env_blocked.set_cell(0, 0, -1)
assert dijkstra(env_blocked, (0, 0), (2, 2)).to_dict() == {
    "path": [], "cost": math.inf, "nodes_explored": 0, "visited_order": []
}
assert a_star(env_blocked, (0, 0), (2, 2)).to_dict() == {
    "path": [], "cost": math.inf, "nodes_explored": 0, "visited_order": []
}
print("Prueba 3 OK (Bloqueos absolutos): manejo limpio sin ciclos (path=[], cost=inf)")
print("Todas las pruebas del Módulo 2 pasaron exitosamente.")


Prueba 1 OK (Rodeo seguro): costo óptimo = 6.00 vs ruta recta riesgosa = 35.18 | pasos = 6
Prueba 2a OK (Ortogonal): costo Dijkstra = 16.00, A* = 16.00 | explorados Dijkstra = 86, A* = 17
Prueba 2b OK (Diagonal):  costo Dijkstra = 13.66, A* = 13.66 | explorados Dijkstra = 89, A* = 38
Prueba 3 OK (Bloqueos absolutos): manejo limpio sin ciclos (path=[], cost=inf)
Todas las pruebas del Módulo 2 pasaron exitosamente.


## Módulo 3 — Suite de Escenarios de Prueba y Auditoría Caza-Errores (Andrey Lechuga)

Esta sección implementa y verifica de forma automatizada los **5 escenarios obligatorios** de la actividad más **1 escenario adicional caza-errores**, ejecutando tanto `dijkstra` como `a_star` sobre cada configuración del terreno y almacenando todas las salidas en el diccionario global `ESCENARIOS_PRUEBA` para su consumo directo por los módulos de Visualización (Módulo 4) y Documentación (Módulo 5).

### Resumen de escenarios implementados
1. **Escenario 1 — Caso Base (`E1_Base`):** Cuadrícula $10\times 10$ con inundaciones parciales y dos pasos sobre un canal anegado. Verifica que ambos algoritmos encuentran la ruta óptima hacia el refugio con el mismo costo acumulado $J(P)$, mientras que $A^*$ explora menos nodos que Dijkstra.
2. **Escenario 2 — Bloqueo Dinámico en Ruta Previa (`E2_Bloqueo_Dinamico`):** Sobre el mismo mapa del Escenario 1, el puente principal `(4, 4)` (que formaba parte de la ruta óptima del Escenario 1) sufre una inundación crítica (`set_cell(4, 4, -1)`). El sistema recalcula dinámicamente sin reconstruir el entorno y desvía la evacuación por el puente secundario oriental.
3. **Escenario 3 — Refugio Inaccesible (`E3_Inaccesible`):** El nivel del agua bloquea también el paso secundario (`set_cell(4, 8, -1)`), aislando por completo la zona sur. Ambos algoritmos agotan la componente conexa norte y finalizan limpiamente reportando `path = []` y `cost = inf` sin entrar en bucles infinitos.
4. **Escenario 4 — Cambio Dinámico de Riesgo (`E4A_Riesgo_Leve` vs. `E4B_Riesgo_Severo`):** Demuestra cómo la actualización del riesgo de una región mediante `set_region_risk` modifica la toma de decisiones del agente:
   - **Antes (`4A`, corredor central con riesgo leve $R=1$):** El algoritmo decide atravesar el corredor en línea recta ($9$ pasos, $J(P)=13.00$) porque es más barato que rodear en seco ($15$ pasos, $J(P)=15.00$).
   - **Después (`4B`, el corredor sube a riesgo severo $R=3$):** El costo del corredor se dispara a $J(P)=30.78$, por lo que el algoritmo cambia automáticamente de decisión y toma el rodeo seco de $15$ pasos con $J(P)=15.00$.
5. **Escenario 5 — El Camino Más Corto vs. El Más Conveniente (`E5_Corto_vs_Conveniente`):** Responde empíricamente a la **pregunta eje de la rúbrica**. Frente al atajo físico directo de **$9$ pasos** que atraviesa agua severa ($R=3$, $J(P)=71.35$), el sistema elige la ruta segura de **$13$ pasos** por terreno seco ($R=0$, $J(P)=13.00$), logrando un **81.8% de reducción en el costo total de evacuación**.
6. **Escenario 6 — Auditoría Caza-Errores (`E6_Caza_Errores`):** Prueba de estrés en tres partes:
   - **6.1 Trampa de relajación tardía en vado inundado ($R=3$):** Con movimiento diagonal activo (`allow_diagonal=True`), un nodo cercano `(1, 1)` ($g=1.41$) descubre primero el único vado `(2, 2)` en diagonal pagando $\sqrt{2}\times 11.39 = 16.11$ ($g_{\text{total}}=17.53$). Un nodo ortogonal `(2, 1)` ($g=2.41$) se expande **después**, pero entra al vado ortogonalmente ($d=1.0$) reduciendo el costo a $2.41 + 11.39 = 13.81$. Esta prueba atrapa el clásico bug de marcar nodos como visitados al encolarlos por primera vez.
   - **6.2 Asimetría direccional de evacuación ($C(u\to v) \neq C(v\to u)$):** Verifica que evacuar desde una celda inundada hacia terreno seco cuesta mucho menos que internarse desde zona seca hacia una celda severamente inundada.
   - **6.3 Auditoría de validaciones en `GridEnvironment.cost(u, v)` y heurística corrupta:** Confirma que las correcciones aplicadas al método `cost()` detectan orígenes fuera de rango, orígenes bloqueados y diagonales no permitidas, y que una heurística no admisible rompe la optimalidad de $A^*$.

In [5]:
# ==============================================================================
# MÓDULO 3: SUITE DE 6 ESCENARIOS DE PRUEBA (5 OBLIGATORIOS + 1 CAZA-ERRORES)
# Autor: Andrey Lechuga
# ==============================================================================

def calcular_costo_ruta(env: GridEnvironment, path: list[Node]) -> float:
    """Recalcula el costo acumulado J(P) paso a paso con env.cost(u, v) y verifica su validez."""
    if not path:
        return math.inf
    total = 0.0
    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        assert env.in_bounds(*u) and env.in_bounds(*v), f"Paso fuera de límites: {u} -> {v}"
        assert not env.is_blocked(*u) and not env.is_blocked(*v), f"Paso por celda bloqueada: {u} -> {v}"
        total += env.cost(u, v)
    return total


def construir_escenario_base_10x10() -> tuple[GridEnvironment, Node, Node]:
    """Construye el mapa base 10x10 para los Escenarios 1, 2 y 3."""
    env = GridEnvironment(10, 10, alpha=2.0, beta=1.5, allow_diagonal=False)
    env.set_region_risk(4, 4, 0, 7, -1)   # canal desbordado (bloqueado)
    env.set_cell(4, 4, 1)                 # puente principal con anegación leve (R=1)
    env.set_region_risk(1, 3, 2, 3, 1)    # charcos leves al norte
    env.set_region_risk(6, 7, 4, 6, 2)    # zona de anegación moderada al sur
    env.set_region_risk(2, 6, 8, 9, 1)    # desvío oriental con anegación leve
    return env, (0, 0), (9, 9)


# Diccionario global para consumo de Módulo 4 (Visualización) y Módulo 5 (Documentación)
ESCENARIOS_PRUEBA: dict[str, dict] = {}
HISTORIAL_METRICAS: list[dict] = []


def ejecutar_escenario(
    clave: str,
    titulo: str,
    env: GridEnvironment,
    start: Node,
    goal: Node,
    descripcion: str = "",
) -> tuple[SearchResult, SearchResult]:
    """Ejecuta Dijkstra y A* en el escenario dado, audita invariantes y guarda métricas reales."""
    res_d = dijkstra(env, start, goal)
    res_a = a_star(env, start, goal)

    # Invariante 1: Optimalidad idéntica (heurística admisible y consistente)
    if math.isinf(res_d.cost):
        assert math.isinf(res_a.cost), f"[{clave}] A* debe reportar inf si Dijkstra reporta inf"
        assert res_d.path == [] and res_a.path == [], f"[{clave}] Ruta debe ser vacía cuando cost=inf"
    else:
        assert math.isclose(res_d.cost, res_a.cost, rel_tol=1e-9), (
            f"[{clave}] Discrepancia de costo óptimo: Dijkstra={res_d.cost} vs A*={res_a.cost}"
        )
        assert math.isclose(calcular_costo_ruta(env, res_d.path), res_d.cost, rel_tol=1e-9)
        assert math.isclose(calcular_costo_ruta(env, res_a.path), res_a.cost, rel_tol=1e-9)
        assert res_d.path[0] == start and res_d.path[-1] == goal
        assert res_a.path[0] == start and res_a.path[-1] == goal

    # Invariante 2: A* explora menor o igual cantidad de nodos que Dijkstra
    assert res_a.nodes_explored <= res_d.nodes_explored, (
        f"[{clave}] A* exploró más nodos ({res_a.nodes_explored}) que Dijkstra ({res_d.nodes_explored})"
    )

    ESCENARIOS_PRUEBA[clave] = {
        "titulo": titulo,
        "descripcion": descripcion,
        "grid_snapshot": env.grid.copy(),
        "allow_diagonal": env.allow_diagonal,
        "start": start,
        "goal": goal,
        "dijkstra": res_d,
        "a_star": res_a,
    }

    for nombre_alg, res in [("Dijkstra", res_d), ("A*", res_a)]:
        HISTORIAL_METRICAS.append({
            "clave": clave,
            "escenario": titulo,
            "algoritmo": nombre_alg,
            "found": res.found,
            "cost": res.cost,
            "steps": res.steps,
            "nodes_explored": res.nodes_explored,
            "time_ms": res.time_ms,
            "path": res.path,
        })

    estado = f"Costo J(P) = {res_a.cost:.2f} | Pasos = {res_a.steps}" if res_a.found else "SIN RUTA (cost = inf)"
    ahorro_nodos = (
        100.0 * (1.0 - res_a.nodes_explored / res_d.nodes_explored)
        if res_d.nodes_explored > 0 else 0.0
    )
    print(f"[{titulo}]")
    print(f"  -> Resultado: {estado}")
    print(
        f"  -> Nodos explorados: Dijkstra = {res_d.nodes_explored} vs A* = {res_a.nodes_explored} "
        f"(reducción del {ahorro_nodos:.1f}% con A*)"
    )
    return res_d, res_a


# ------------------------------------------------------------------------------
# ESCENARIO 1 (Obligatorio): Caso base — Ruta válida disponible hacia el refugio
# ------------------------------------------------------------------------------
env_dinamico, start_base, goal_base = construir_escenario_base_10x10()
res_d_e1, res_a_e1 = ejecutar_escenario(
    "E1_Base",
    "Escenario 1: Ruta base disponible",
    env_dinamico,
    start_base,
    goal_base,
    "Terreno 10x10 con puente principal abierto en (4,4).",
)
assert (4, 4) in res_a_e1.path, "La ruta base debe cruzar el puente principal (4,4)"

# ------------------------------------------------------------------------------
# ESCENARIO 2 (Obligatorio): Bloqueo dinámico en la ruta previa y recálculo
# ------------------------------------------------------------------------------
# Bloqueamos dinámicamente el puente principal (4,4) por donde pasaba la ruta del Escenario 1
env_dinamico.set_cell(4, 4, -1)
res_d_e2, res_a_e2 = ejecutar_escenario(
    "E2_Bloqueo_Dinamico",
    "Escenario 2: Bloqueo dinámico en ruta previa",
    env_dinamico,
    start_base,
    goal_base,
    "Se inunda dinámicamente el puente (4,4); el agente recalcula hacia el paso oriental (col 8-9).",
)
assert (4, 4) not in res_a_e2.path, "La nueva ruta no puede cruzar la celda bloqueada (4,4)"
assert res_a_e2.cost > res_a_e1.cost, "El bloqueo del puente principal incrementa el costo óptimo"
print(f"  -> Verificado: Al colapsar (4,4), el sistema desvió la ruta por el paso oriental (costo {res_a_e1.cost:.2f} -> {res_a_e2.cost:.2f}).\n")

# ------------------------------------------------------------------------------
# ESCENARIO 3 (Obligatorio): Refugio completamente inaccesible
# ------------------------------------------------------------------------------
# Bloqueamos también el paso oriental en (4,8) y (4,9) cerrando toda la fila 4
env_dinamico.set_region_risk(4, 4, 8, 9, -1)
res_d_e3, res_a_e3 = ejecutar_escenario(
    "E3_Inaccesible",
    "Escenario 3: Refugio completamente inaccesible",
    env_dinamico,
    start_base,
    goal_base,
    "Colapso total de los pasos norte-sur en la fila 4; el refugio (9,9) queda aislado.",
)
assert not res_a_e3.found and math.isinf(res_a_e3.cost) and res_a_e3.path == []
print("  -> Verificado: Ambos algoritmos detectan inaccesibilidad (path=[], cost=inf) sin ciclarse.\n")

# ------------------------------------------------------------------------------
# ESCENARIO 4 (Obligatorio): Cambio dinámico en el nivel de riesgo de una región
# ------------------------------------------------------------------------------
env_e4 = GridEnvironment(8, 10, alpha=2.0, beta=1.5, allow_diagonal=False)
start_e4, goal_e4 = (3, 0), (3, 9)
# Muros que canalizan el flujo entre el corredor central (fila 3) y el rodeo seco sur (fila 6)
env_e4.set_region_risk(0, 2, 2, 7, -1)
env_e4.set_region_risk(4, 5, 2, 7, -1)

# 4A: Antes — El corredor central tiene anegación leve (R=1 en cols 4..5)
env_e4.set_region_risk(3, 3, 4, 5, 1)
res_d_e4a, res_a_e4a = ejecutar_escenario(
    "E4A_Riesgo_Leve",
    "Escenario 4A: Antes (corredor con riesgo leve R=1)",
    env_e4,
    start_e4,
    goal_e4,
    "Corredor central con riesgo leve R=1: conviene cruzar directo (9 pasos, costo 13.00).",
)

# 4B: Después — Sube el nivel del agua en la región central a riesgo severo (R=3)
env_e4.set_region_risk(3, 3, 3, 6, 3)
res_d_e4b, res_a_e4b = ejecutar_escenario(
    "E4B_Riesgo_Severo",
    "Escenario 4B: Después (corredor sube a riesgo severo R=3)",
    env_e4,
    start_e4,
    goal_e4,
    "El corredor sube a R=3: el sistema cambia de decisión y rodea por la zona seca sur (15 pasos, costo 15.00).",
)
assert res_a_e4a.steps == 9 and math.isclose(res_a_e4a.cost, 13.0)
assert res_a_e4b.steps == 15 and math.isclose(res_a_e4b.cost, 15.0)
assert res_a_e4a.path != res_a_e4b.path
print(
    f"  -> Verificado: Con R=1 cruzó directo ({res_a_e4a.steps} pasos, J={res_a_e4a.cost:.2f}); "
    f"al subir a R=3 cambió al rodeo seco ({res_a_e4b.steps} pasos, J={res_a_e4b.cost:.2f}).\n"
)

# ------------------------------------------------------------------------------
# ESCENARIO 5 (Obligatorio): Camino más corto vs. camino más conveniente (Pregunta Eje)
# ------------------------------------------------------------------------------
env_e5 = GridEnvironment(7, 10, alpha=2.0, beta=1.5, allow_diagonal=False)
start_e5, goal_e5 = (3, 0), (3, 9)
# El camino más corto en pasos (fila 3 recta, 9 pasos) cruza 6 celdas de inundación severa R=3
env_e5.set_region_risk(3, 3, 2, 7, 3)
# Separadores para contrastar limpiamente el atajo inundado contra el rodeo seco por fila 1
env_e5.set_region_risk(2, 2, 2, 7, -1)
env_e5.set_region_risk(4, 4, 2, 7, -1)

ruta_mas_corta_fisica = [(3, c) for c in range(10)]  # 9 pasos exactos en línea recta
costo_ruta_mas_corta = calcular_costo_ruta(env_e5, ruta_mas_corta_fisica)

res_d_e5, res_a_e5 = ejecutar_escenario(
    "E5_Corto_vs_Conveniente",
    "Escenario 5: Camino más corto vs. más conveniente (Pregunta Eje)",
    env_e5,
    start_e5,
    goal_e5,
    "Demostración de la pregunta eje: el sistema descarta la ruta de 9 pasos (J=71.35) y elige la de 13 pasos (J=13.00).",
)
assert res_a_e5.steps > len(ruta_mas_corta_fisica) - 1
assert res_a_e5.cost < costo_ruta_mas_corta
ESCENARIOS_PRUEBA["E5_Corto_vs_Conveniente"]["comparativa_pregunta_eje"] = {
    "ruta_mas_corta_pasos": len(ruta_mas_corta_fisica) - 1,
    "ruta_mas_corta_costo": costo_ruta_mas_corta,
    "ruta_mas_corta_path": ruta_mas_corta_fisica,
    "ruta_elegida_pasos": res_a_e5.steps,
    "ruta_elegida_costo": res_a_e5.cost,
    "ruta_elegida_path": res_a_e5.path,
}
print(
    f"  -> Demostración Pregunta Eje:\n"
    f"     * Ruta más corta en distancia: {len(ruta_mas_corta_fisica) - 1} pasos, pero Costo J(P) = {costo_ruta_mas_corta:.2f} (cruza R=3)\n"
    f"     * Ruta más conveniente elegida: {res_a_e5.steps} pasos (rodeo seco), con Costo J(P) = {res_a_e5.cost:.2f} "
    f"(¡{100.0 * (1.0 - res_a_e5.cost / costo_ruta_mas_corta):.1f}% menor costo acumulado!)\n"
)

# ------------------------------------------------------------------------------
# ESCENARIO 6 (Adicional Caza-Errores):
# 6.1 Trampa de Relajación Tardía en Vado Inundado (R=3) con Diagonales
# 6.2 Asimetría Direccional del Costo de Evacuación
# 6.3 Auditoría de Validaciones en GridEnvironment.cost(u, v) y Admisibilidad
# ------------------------------------------------------------------------------
print("--- Ejecutando Escenario 6: Auditoría Caza-Errores ---")
env_e6 = GridEnvironment(4, 4, alpha=2.0, beta=1.5, allow_diagonal=True)
env_e6.set_region_risk(0, 3, 2, 2, -1)  # muralla vertical bloqueada en la columna 2
env_e6.set_cell(2, 2, 3)                # único vado transitable en (2,2) con riesgo severo R=3
start_e6, goal_e6 = (0, 0), (2, 3)

res_d_e6, res_a_e6 = ejecutar_escenario(
    "E6_Caza_Errores",
    "Escenario 6: Caza-errores (Relajación tardía en vado R=3)",
    env_e6,
    start_e6,
    goal_e6,
    "Verifica que la entrada ortogonal tardía (d=1.0) desde (2,1) relaje la entrada diagonal temprana (d=sqrt(2)) desde (1,1).",
)

# Verificación 6.1: La ruta óptima DEBE entrar al vado (2,2) ortogonalmente desde (2,1) y no en diagonal desde (1,1)
idx_vado = res_a_e6.path.index((2, 2))
nodo_previo_vado = res_a_e6.path[idx_vado - 1]
assert nodo_previo_vado == (2, 1), f"Debía entrar ortogonalmente desde (2,1), entró desde {nodo_previo_vado}"


def _simular_bug_visitado_al_encolar(env: GridEnvironment, start: Node, goal: Node) -> float:
    """Simula el error común de marcar un nodo como visitado en cuanto se inserta al heap,
    impidiendo que un camino descubierto después (como (2,1) -> (2,2) ortogonal) relaje su costo."""
    heap = [(0.0, start)]
    visitados_al_encolar = {start}
    while heap:
        costo_actual, u = heapq.heappop(heap)
        if u == goal:
            return costo_actual
        for v, step_cost in env.get_neighbors(u):
            if v not in visitados_al_encolar:
                visitados_al_encolar.add(v)
                heapq.heappush(heap, (costo_actual + step_cost, v))
    return math.inf


costo_con_bug_relajacion = _simular_bug_visitado_al_encolar(env_e6, start_e6, goal_e6)
assert costo_con_bug_relajacion > res_a_e6.cost
print(
    f"  -> [6.1 Relajación Tardía OK] Entrar ortogonalmente por (2,1)->(2,2) dio J(P) = {res_a_e6.cost:.2f}; "
    f"un algoritmo con bug de visitado prematuro entra en diagonal (1,1)->(2,2) pagando J(P) = {costo_con_bug_relajacion:.2f} "
    f"(+{costo_con_bug_relajacion - res_a_e6.cost:.2f} de sobrecosto)."
)

# Verificación 6.2: Asimetría direccional C(u -> v) != C(v -> u)
env_asim = GridEnvironment(1, 4, alpha=2.0, beta=1.5)
env_asim.set_cell(0, 0, 3)  # Casa inicial inundada (R=3), refugio en (0,3) seco (R=0)
res_evacuar = a_star(env_asim, (0, 0), (0, 3))
res_internarse = a_star(env_asim, (0, 3), (0, 0))
assert math.isclose(res_evacuar.cost, 3.0)
assert math.isclose(res_internarse.cost, 2.0 + (1.0 + 2.0 * (3 ** 1.5)))
assert res_evacuar.cost < res_internarse.cost
print(
    f"  -> [6.2 Asimetría Direccional OK] Evacuar (0,0)[R=3] -> (0,3)[R=0] cuesta J = {res_evacuar.cost:.2f}, "
    f"mientras que el sentido inverso hacia la zona inundada cuesta J = {res_internarse.cost:.2f}."
)

# Verificación 6.3: Auditoría de validaciones corregidas en GridEnvironment.cost(u, v) y heurística sobreestimada
env_val = GridEnvironment(4, 4, allow_diagonal=False)
env_val.set_cell(0, 0, -1)
assert math.isinf(env_val.cost((0, 0), (0, 1))), "Origen bloqueado debe devolver costo infinito"
try:
    env_val.cost((-1, 0), (0, 0))
    raise AssertionError("Debió lanzar IndexError para origen fuera de límites")
except IndexError:
    pass
try:
    env_val.cost((1, 1), (2, 2))
    raise AssertionError("Debió lanzar ValueError para diagonal con allow_diagonal=False")
except ValueError:
    pass

# Heurística corrupta (no admisible) que infla 50x la distancia vertical y engaña a A* en el Escenario 5
res_a_corrupta = a_star(env_e5, start_e5, goal_e5, heuristic_fn=lambda u, g: 50.0 * abs(u[0] - g[0]) + abs(u[1] - g[1]))
assert res_a_corrupta.cost > res_d_e5.cost, "Una heurística no admisible debe perder la optimalidad frente a Dijkstra"
print(
    f"  -> [6.3 Validaciones y Heurística OK] Fronteras/diagonales validadas en env.cost(u,v) y comprobado que una "
    f"heurística no admisible degrada A* (J = {res_a_corrupta.cost:.2f} vs óptimo J = {res_d_e5.cost:.2f})."
)

[Escenario 1: Ruta base disponible]
  -> Resultado: Costo J(P) = 20.00 | Pasos = 18
  -> Nodos explorados: Dijkstra = 89 vs A* = 44 (reducción del 50.6% con A*)
[Escenario 2: Bloqueo dinámico en ruta previa]
  -> Resultado: Costo J(P) = 26.00 | Pasos = 18
  -> Nodos explorados: Dijkstra = 66 vs A* = 48 (reducción del 27.3% con A*)
  -> Verificado: Al colapsar (4,4), el sistema desvió la ruta por el paso oriental (costo 20.00 -> 26.00).

[Escenario 3: Refugio completamente inaccesible]
  -> Resultado: SIN RUTA (cost = inf)
  -> Nodos explorados: Dijkstra = 40 vs A* = 40 (reducción del 0.0% con A*)
  -> Verificado: Ambos algoritmos detectan inaccesibilidad (path=[], cost=inf) sin ciclarse.

[Escenario 4A: Antes (corredor con riesgo leve R=1)]
  -> Resultado: Costo J(P) = 13.00 | Pasos = 9
  -> Nodos explorados: Dijkstra = 42 vs A* = 14 (reducción del 66.7% con A*)
[Escenario 4B: Después (corredor sube a riesgo severo R=3)]
  -> Resultado: Costo J(P) = 15.00 | Pasos = 15
  -> Nodos explor

In [6]:
# ==============================================================================
# RESUMEN TABULAR DE LA SUITE DE ESCENARIOS (MÓDULO 3)
# ==============================================================================
print("=" * 102)
print(f"{'ESCENARIO DE PRUEBA':<52} | {'ALG.':<8} | {'COSTO J(P)':>10} | {'PASOS':>5} | {'NODOS':>5} | {'TIEMPO':>9}")
print("=" * 102)
for fila in HISTORIAL_METRICAS:
    costo_str = f"{fila['cost']:.2f}" if fila["found"] else "inf"
    pasos_str = str(fila["steps"]) if fila["found"] else "-"
    print(
        f"{fila['escenario'][:52]:<52} | {fila['algoritmo']:<8} | {costo_str:>10} | "
        f"{pasos_str:>5} | {fila['nodes_explored']:>5} | {fila['time_ms']:>6.3f} ms"
    )
print("=" * 102)
print(f"Total de ejecuciones verificadas: {len(HISTORIAL_METRICAS)} ({len(ESCENARIOS_PRUEBA)} escenarios x 2 algoritmos) — 100% asserts superados.")

ESCENARIO DE PRUEBA                                  | ALG.     | COSTO J(P) | PASOS | NODOS |    TIEMPO
Escenario 1: Ruta base disponible                    | Dijkstra |      20.00 |    18 |    89 |  0.570 ms
Escenario 1: Ruta base disponible                    | A*       |      20.00 |    18 |    44 |  0.299 ms
Escenario 2: Bloqueo dinámico en ruta previa         | Dijkstra |      26.00 |    18 |    66 |  0.414 ms
Escenario 2: Bloqueo dinámico en ruta previa         | A*       |      26.00 |    18 |    48 |  0.326 ms
Escenario 3: Refugio completamente inaccesible       | Dijkstra |        inf |     - |    40 |  0.251 ms
Escenario 3: Refugio completamente inaccesible       | A*       |        inf |     - |    40 |  0.265 ms
Escenario 4A: Antes (corredor con riesgo leve R=1)   | Dijkstra |      13.00 |     9 |    42 |  0.242 ms
Escenario 4A: Antes (corredor con riesgo leve R=1)   | A*       |      13.00 |     9 |    14 |  0.081 ms
Escenario 4B: Después (corredor sube a riesgo severo | 

## Módulo 4: Visualización en Mapas de Calor y Gráficas Comparativas (Izquierdo)

Este módulo implementa la capa de presentación y análisis visual del sistema de evacuación inteligente:
1. **Mapas de calor categóricos del terreno:** Representación visual de los niveles de anegación del terreno ($R=0$ seco, $R=1$ leve, $R=2$ moderado, $R=3$ severo y $-1$ bloqueo total) con paletas accesibles de alto contraste.
2. **Superposición de rutas óptimas:** Trazado de las trayectorias de evacuación calculadas por Dijkstra y $A^*$, con identificación clara del punto de partida $\mathbf{S}$ (Start) y el refugio seguro $\mathbf{G}$ (Goal).
3. **Visualización de la Pregunta Eje (Escenario 5):** Comparación simultánea en el mismo mapa entre la **ruta más corta física** (que atraviesa imprudentemente agua profunda $R=3$) y la **ruta más conveniente** (el rodeo seco que minimiza el costo acumulado $J(P)$).
4. **Dashboard comparativo de métricas:** Gráficas de barras de nodos explorados (Dijkstra vs $A^*$), verificación de optimalidad de costos y tiempos de ejecución.


In [ ]:
# ==============================================================================
# MÓDULO 4: VISUALIZACIÓN EN MAPAS DE CALOR Y RUTAS DE EVACUACIÓN
# Autor: Izquierdo
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

# Paleta categórica personalizada para los niveles de riesgo
# -1: Bloqueo (Gris pizarra oscuro)
#  0: Seco (Verde menta claro / transitable seguro)
#  1: Anegación leve (Azul cielo / agua baja)
#  2: Anegación moderada (Ámbar / naranja)
#  3: Anegación severa (Rojo carmesí / agua profunda peligrosa)
COLORES_RIESGO = {
    -1: "#1E293B",  # Slate oscuro (Bloqueo)
     0: "#DCFCE7",  # Verde menta claro (Seco R=0)
     1: "#BAE6FD",  # Azul cielo claro (Leve R=1)
     2: "#FDBA74",  # Naranja claro (Moderado R=2)
     3: "#EF4444",  # Rojo intenso (Severo R=3)
}

CAMPOS_VALORES = [-1, 0, 1, 2, 3]
CMAP_RIESGO = mcolors.ListedColormap([COLORES_RIESGO[v] for v in CAMPOS_VALORES])
NORM_RIESGO = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5, 3.5], CMAP_RIESGO.N)

ETIQUETAS_LEYENDA = [
    mpatches.Patch(color=COLORES_RIESGO[-1], label="Bloqueo (-1)"),
    mpatches.Patch(color=COLORES_RIESGO[0],  label="Seco (R=0, 1.0×)"),
    mpatches.Patch(color=COLORES_RIESGO[1],  label="Leve (R=1, 3.0×)"),
    mpatches.Patch(color=COLORES_RIESGO[2],  label="Moderado (R=2, 6.7×)"),
    mpatches.Patch(color=COLORES_RIESGO[3],  label="Severo (R=3, 11.4×)"),
]


def plot_grid_and_path(
    grid: np.ndarray,
    start: tuple[int, int],
    goal: tuple[int, int],
    path: list[tuple[int, int]] | None = None,
    path_color: str = "#2563EB",
    path_label: str = "Ruta Óptima",
    secondary_path: list[tuple[int, int]] | None = None,
    secondary_color: str = "#DC2626",
    secondary_label: str = "Ruta Alternativa",
    title: str = "",
    subtitle: str = "",
    ax: plt.Axes | None = None,
    show_grid_lines: bool = True,
    show_legend: bool = False,
) -> plt.Axes:
    """Dibuja un mapa de calor del terreno con inicio, meta y rutas de evacuación superpuestas."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    rows, cols = grid.shape

    # 1. Renderizar mapa de calor del terreno
    ax.imshow(grid, cmap=CMAP_RIESGO, norm=NORM_RIESGO, origin="upper")

    # 2. Cuadrícula sutil
    if show_grid_lines:
        ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
        ax.grid(which="minor", color="#94A3B8", linestyle=":", linewidth=0.6, alpha=0.5)
        ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.tick_params(axis="both", labelsize=8)

    # 3. Marcar celdas bloqueadas con una 'X' sutil
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == -1:
                ax.text(c, r, "✕", color="#94A3B8", fontsize=10, ha="center", va="center", weight="bold")

    # 4. Trazar ruta secundaria si existe (ej. ruta corta física en E5)
    if secondary_path and len(secondary_path) > 1:
        s_rows, s_cols = zip(*secondary_path)
        ax.plot(s_cols, s_rows, color=secondary_color, linewidth=2.8, linestyle="--",
                marker="s", markersize=4, label=secondary_label, zorder=3, alpha=0.9)

    # 5. Trazar ruta principal (A* / Dijkstra)
    if path and len(path) > 1:
        p_rows, p_cols = zip(*path)
        ax.plot(p_cols, p_rows, color=path_color, linewidth=3.2, linestyle="-",
                marker="o", markersize=5, label=path_label, zorder=4)
    elif path is not None and len(path) == 0:
        # Indicador visual de ruta inaccesible
        ax.text(
            cols / 2 - 0.5, rows / 2 - 0.5,
            "SIN RUTA DE EVACUACIÓN\n(Refugio Inaccesible)",
            color="#FFFFFF", fontsize=9, weight="bold", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#DC2626", edgecolor="#7F1D1D", alpha=0.9),
            zorder=6
        )

    # 6. Marcadores destacados de Inicio (S) y Refugio (G)
    sr, sc = start
    gr, gc = goal
    ax.scatter(sc, sr, s=170, color="#16A34A", edgecolor="#FFFFFF", linewidth=1.5, zorder=5)
    ax.text(sc, sr, "S", color="#FFFFFF", fontsize=9, weight="bold", ha="center", va="center", zorder=6)

    ax.scatter(gc, gr, s=200, color="#F59E0B", edgecolor="#1E293B", linewidth=1.5, marker="*", zorder=5)
    ax.text(gc, gr + 0.35, "Refugio", color="#1E293B", fontsize=7.5, weight="bold", ha="center", va="top", zorder=6)

    titulo_completo = f"{title}\n{subtitle}" if subtitle else title
    ax.set_title(titulo_completo, fontsize=9.5, fontweight="bold", pad=7)

    if show_legend:
        rutas_legend = []
        if secondary_path:
            rutas_legend.append(mpatches.Patch(color=secondary_color, label=secondary_label))
        if path:
            rutas_legend.append(mpatches.Patch(color=path_color, label=path_label))
        ax.legend(handles=ETIQUETAS_LEYENDA + rutas_legend, loc="upper left", bbox_to_anchor=(1.03, 1), fontsize=8)

    return ax


# Renderizar los 6 Escenarios en un Mosaico 2x3 de Mapas de Calor
claves_escenarios = ["E1_Base", "E2_Bloqueo_Dinamico", "E3_Inaccesible", "E4A_Riesgo_Leve", "E4B_Riesgo_Severo", "E6_Caza_Errores"]
titulos_cortos = [
    ("E1: Ruta Base (Puente Central)", "Cruza por paso R=1"),
    ("E2: Bloqueo Dinámico", "Colapso (4,4) -> Desvío"),
    ("E3: Inaccesibilidad Total", "Canal cerrado (sin ruta)"),
    ("E4A: Corredor Leve (R=1)", "Cruce directo óptimo"),
    ("E4B: Corredor Severo (R=3)", "Cambio a rodeo seguro"),
    ("E6: Caza-Errores (Vado)", "Relajación ortogonal R=3"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10.5))
fig.suptitle("MÓDULO 4: MAPAS DE CALOR Y RUTAS DE EVACUACIÓN ÓPTIMAS (A*)\nFundamentos de IA — Práctica 2: Cuando el camino más corto no es el mejor",
             fontsize=13, fontweight="bold", y=0.98)

for ax, clave, (tit, subtit) in zip(axes.flat, claves_escenarios, titulos_cortos):
    info = ESCENARIOS_PRUEBA[clave]
    res_a = info["a_star"]
    costo_str = f"Costo J={res_a.cost:.2f} | Pasos={res_a.steps}" if res_a.found else "Costo J=inf | Sin camino"
    sub_txt = f"{subtit} | {costo_str}"

    plot_grid_and_path(
        grid=info["grid_snapshot"],
        start=info["start"],
        goal=info["goal"],
        path=res_a.path,
        path_color="#2563EB",
        path_label="Ruta Óptima A*",
        title=tit,
        subtitle=sub_txt,
        ax=ax,
        show_grid_lines=True,
        show_legend=False
    )

# Leyenda unificada al pie de la figura
fig.legend(handles=ETIQUETAS_LEYENDA + [
    mpatches.Patch(color="#2563EB", label="Ruta Óptima de Evacuación"),
    mpatches.Patch(color="#16A34A", label="S: Inicio"),
    mpatches.Patch(color="#F59E0B", label="*: Refugio"),
], loc="lower center", ncol=8, bbox_to_anchor=(0.5, 0.01), fontsize=9, frameon=True)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


In [ ]:
# ==============================================================================
# DEMOSTRACIÓN VISUAL DE LA PREGUNTA EJE (ESCENARIO 5: CORTO VS CONVENIENTE)
# "¿Si dos rutas llegan al mismo refugio, pero una es más corta y la otra
#  atraviesa zonas de mayor riesgo, cómo determina el sistema cuál es la mejor?"
# ==============================================================================

e5_info = ESCENARIOS_PRUEBA["E5_Corto_vs_Conveniente"]
cmp_e5 = e5_info["comparativa_pregunta_eje"]

fig, (ax_map, ax_bar) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.1, 1]})
fig.suptitle("PREGUNTA EJE: ¿POR QUÉ EL CAMINO MÁS CORTO NO ES EL MEJOR?\nAnálisis Comparativo del Escenario 5",
             fontsize=13, fontweight="bold", y=0.98)

# Panel 1: Mapa con ambas rutas superpuestas
plot_grid_and_path(
    grid=e5_info["grid_snapshot"],
    start=e5_info["start"],
    goal=e5_info["goal"],
    path=cmp_e5["ruta_elegida_path"],
    path_color="#2563EB",
    path_label=f"Ruta Conveniente ({cmp_e5['ruta_elegida_pasos']} pasos, J={cmp_e5['ruta_elegida_costo']:.1f})",
    secondary_path=cmp_e5["ruta_mas_corta_path"],
    secondary_color="#DC2626",
    secondary_label=f"Ruta Corta Física ({cmp_e5['ruta_mas_corta_pasos']} pasos, J={cmp_e5['ruta_mas_corta_costo']:.1f})",
    title="Mapa de Evacuación: Rodeo Seco vs. Cruce Inundado",
    subtitle="La ruta roja cruza 4 celdas R=3 (11.4× c/u). La ruta azul rodea por R=0 (1.0×).",
    ax=ax_map,
    show_legend=True
)

# Panel 2: Comparativa cuantitativa de Costo vs Pasos
nombres_rutas = ["Ruta Corta Física\n(Línea Recta)", "Ruta Elegida\n(Rodeo Seguro A*)"]
pasos_valores = [cmp_e5["ruta_mas_corta_pasos pioneers" if "ruta_mas_corta_pasos pioneers" in cmp_e5 else "ruta_mas_corta_pasos"], cmp_e5["ruta_elegida_pasos"]]
costo_valores = [cmp_e5["ruta_mas_corta_costo"], cmp_e5["ruta_elegida_costo"]]

x = np.arange(len(nombres_rutas))
w = 0.35

rects_cost = ax_bar.bar(x - w/2, costo_valores, w, label="Costo Acumulado J(P) [Menor es Mejor]",
                        color=["#EF4444", "#10B981"], edgecolor="#1E293B", linewidth=1.2)
rects_step = ax_bar.bar(x + w/2, pasos_valores, w, label="Distancia Física (Número de Pasos)",
                        color=["#FCA5A5", "#6EE7B7"], edgecolor="#1E293B", linewidth=1.2)

ax_bar.set_title("Comparativa de Costo Acumulado J(P) vs Distancia Física", fontsize=11, fontweight="bold", pad=10)
ax_bar.set_xticks(x)
ax_bar.set_xticklabels(nombres_rutas, fontsize=10, fontweight="bold")
ax_bar.set_ylabel("Valor Cuantitativo", fontsize=10, fontweight="bold")
ax_bar.grid(axis="y", linestyle="--", alpha=0.6)
ax_bar.legend(loc="upper right", fontsize=9)

# Anotaciones numéricas sobre las barras
for r in rects_cost:
    h = r.get_height()
    ax_bar.annotate(f"J = {h:.2f}",
                    xy=(r.get_x() + r.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=10, fontweight="bold")

for r in rects_step:
    h = r.get_height()
    ax_bar.annotate(f"{int(h)} pasos",
                    xy=(r.get_x() + r.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

# Badge de ahorro de riesgo
ahorro_pct = 100.0 * (1.0 - cmp_e5["ruta_elegida_costo"] / cmp_e5["ruta_mas_corta_costo"])
ax_bar.text(
    0.5, 0.45,
    f"★ CONCLUSIÓN DE LA PREGUNTA EJE ★\n"
    f"Aunque la ruta elegida tiene +4 pasos más,\n"
    f"reduce el riesgo acumulado en un {ahorro_pct:.1f}%.\n"
    f"El sistema protege vidas minimizando el costo J(P).",
    transform=ax_bar.transAxes, ha="center", va="center",
    fontsize=9.5, fontweight="bold", color="#065F46",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="#ECFDF5", edgecolor="#10B981", linewidth=1.5)
)

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# DASHBOARD DE RENDIMIENTO Y COMPARATIVA: DIJKSTRA VS A*
# ==============================================================================

# Extraer métricas consolidadas
escenarios_unicos = []
dijkstra_nodos = []
astar_nodos = []
dijkstra_costos = []
astar_costos = []
dijkstra_tiempos = []
astar_tiempos = []
reduccion_nodos_pct = []

for esc_clave in ["E1_Base", "E2_Bloqueo_Dinamico", "E3_Inaccesible", "E4A_Riesgo_Leve", "E4B_Riesgo_Severo", "E5_Corto_vs_Conveniente", "E6_Caza_Errores"]:
    d = ESCENARIOS_PRUEBA[esc_clave]["dijkstra"]
    a = ESCENARIOS_PRUEBA[esc_clave]["a_star"]
    
    label_esc = esc_clave.replace("_", "\n")
    escenarios_unicos.append(label_esc)
    
    dijkstra_nodos.append(d.nodes_explored)
    astar_nodos.append(a.nodes_explored)
    
    dijkstra_costos.append(d.cost if d.found else 0)
    astar_costos.append(a.cost if a.found else 0)
    
    dijkstra_tiempos.append(d.time_ms)
    astar_tiempos.append(a.time_ms)
    
    pct = ((d.nodes_explored - a.nodes_explored) / d.nodes_explored * 100) if d.nodes_explored > 0 else 0
    reduccion_nodos_pct.append(pct)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("DASHBOARD COMPARATIVO DE RENDIMIENTO: DIJKSTRA VS A*\nMétricas Consolidadas sobre los 7 Escenarios Verificados",
             fontsize=13, fontweight="bold", y=1.02)

x = np.arange(len(escenarios_unicos))
w = 0.38

# 1. Gráfica de Nodos Explorados
ax1 = axes[0]
rects1 = ax1.bar(x - w/2, dijkstra_nodos, w, label="Dijkstra", color="#3B82F6", edgecolor="#1E293B")
rects2 = ax1.bar(x + w/2, astar_nodos, w, label="A*", color="#10B981", edgecolor="#1E293B")
ax1.set_title("Estados/Nodos Explorados (Menor es Mejor)", fontsize=10.5, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(escenarios_unicos, fontsize=8)
ax1.set_ylabel("Nodos Expandidos", fontsize=9, fontweight="bold")
ax1.grid(axis="y", linestyle="--", alpha=0.6)
ax1.legend(loc="upper right", fontsize=8.5)

# Etiquetas de porcentaje de reducción sobre A*
for i, (pct, r2) in enumerate(zip(reduccion_nodos_pct, rects2)):
    if pct > 0:
        ax1.annotate(f"-{pct:.0f}%",
                     xy=(r2.get_x() + r2.get_width() / 2, r2.get_height()),
                     xytext=(0, 3), textcoords="offset points",
                     ha="center", va="bottom", fontsize=7.5, fontweight="bold", color="#047857")

# 2. Gráfica de Costo Óptimo (Verificación de Admisibilidad)
ax2 = axes[1]
ax2.bar(x - w/2, dijkstra_costos, w, label="Dijkstra", color="#3B82F6", edgecolor="#1E293B")
ax2.bar(x + w/2, astar_costos, w, label="A*", color="#10B981", edgecolor="#1E293B")
ax2.set_title("Costo Óptimo J(P) (Demostración de Admisibilidad)", fontsize=10.5, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(escenarios_unicos, fontsize=8)
ax2.set_ylabel("Costo Acumulado J(P)", fontsize=9, fontweight="bold")
ax2.grid(axis="y", linestyle="--", alpha=0.6)
ax2.legend(loc="upper right", fontsize=8.5)

# 3. Gráfica de Tiempo de Ejecución (ms)
ax3 = axes[2]
ax3.bar(x - w/2, dijkstra_tiempos, w, label="Dijkstra", color="#3B82F6", edgecolor="#1E293B")
ax3.bar(x + w/2, astar_tiempos, w, label="A*", color="#10B981", edgecolor="#1E293B")
ax3.set_title("Tiempo de Ejecución en Milisegundos", fontsize=10.5, fontweight="bold")
ax3.set_xticks(x)
ax3.set_xticklabels(escenarios_unicos, fontsize=8)
ax3.set_ylabel("Tiempo (ms)", fontsize=9, fontweight="bold")
ax3.grid(axis="y", linestyle="--", alpha=0.6)
ax3.legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.show()


### Conclusiones y Hallazgos Visuales (Módulo 4 — Izquierdo)

1. **Evidencia Visual de la Pregunta Eje:**
   - La visualización en el **Escenario 5** demuestra de manera incontestable por qué el camino más corto en distancia no es el mejor en evacuaciones: la ruta directa de 9 pasos atraviesa una zona inundada severa ($R=3$), acumulando un costo de $71.35$. El algoritmo selecciona el rodeo seco de 13 pasos con costo $13.00$, logrando una **reducción del 81.8% en el riesgo acumulado**.
2. **Eficiencia Demostrada de A\*:**
   - En todos los escenarios transitables, $A^*$ reduce los nodos explorados frente a Dijkstra de forma dramática: **-50.6% en E1, -66.7% en E4A, y -67.9% en E5**, manteniendo **exactamente el mismo costo óptimo $J(P)$**. Esto confirma empíricamente que la heurística de distancia base es admisible y consistente.
3. **Robustez ante Inaccesibilidad y Bloqueos Dinámicos:**
   - En el **Escenario 2**, el mapa de calor muestra cómo la ruta se adapta en tiempo real cuando colapsa el paso central `(4, 4)`, redirigiendo el flujo hacia el paso oriental.
   - En el **Escenario 3**, cuando el refugio queda completamente aislado, el visualizador indica el estado *"SIN RUTA DE EVACUACIÓN (Refugio Inaccesible)"* sin caer en ciclos infinitos ni lecturas anómalas.
